In [12]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [13]:
PATH_DIABETES     = '../../diabetesData/diabetes.csv'
PATH_DIABETES_UCI = '../../diabetesData/diabetesUCI.csv'
PATH_GHB          = '../../diabetesData/P_GHB.xpt'
PATH_GLU          = '../../diabetesData/P_GLU.xpt'
PATH_HDL          = '../../diabetesData/P_HDL.xpt'
PATH_INS          = '../../diabetesData/P_INS.xpt'
PATH_TCHOL        = '../../diabetesData/P_TCHOL.xpt'
PATH_TRIGLY       = '../../diabetesData/P_TRIGLY.xpt'
PATH_DEMO         = '../../diabetesData/P_DEMO.xpt'


paths = {
    'diabetes': PATH_DIABETES,
    'diabetes_uci': PATH_DIABETES_UCI,
    'ghb': PATH_GHB,
    'glu': PATH_GLU,
    'hdl': PATH_HDL,
    'ins': PATH_INS,
    'tchol': PATH_TCHOL,
    'trigly': PATH_TRIGLY,
    'demo': PATH_DEMO
}

for name, path in paths.items():
    print(f'{name}:', os.path.exists(path))

diabetes: True
diabetes_uci: True
ghb: True
glu: True
hdl: True
ins: True
tchol: True
trigly: True
demo: True


## Pima / Diabetes 

In [14]:
diabetes = pd.read_csv(PATH_DIABETES)
print('Diabetes dataset shape:', diabetes.shape)
print('Columns:', diabetes.columns.tolist())
print('Missing values:')
print(diabetes.isnull().sum())
diabetes.head()

Diabetes dataset shape: (768, 9)
Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [19]:
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
diabetes[cols] = diabetes[cols].astype(float)

for col in cols:
    for outcome in [0, 1]:
        mask = (diabetes['Outcome'] == outcome) & (diabetes[col] == 0)
        median_val = diabetes.loc[diabetes['Outcome'] == outcome, col].replace(0, np.nan).median()
        diabetes.loc[mask, col] = median_val

diabetes.dropna(inplace=True)
diabetes.drop_duplicates(inplace=True)
print("Final Pima shape:", diabetes.shape)
# cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
# diabetes = diabetes[(diabetes[cols] != 0).all(axis=1)]
# print(diabetes.shape)

Final Pima shape: (768, 9)


In [20]:
diabetes_uci = pd.read_csv(PATH_DIABETES_UCI)
print('Diabetes UCI shape:', diabetes_uci.shape)
print('Columns:', diabetes_uci.columns.tolist())
print('Missing values:')
print(diabetes_uci.isnull().sum())
diabetes_uci.head()

Diabetes UCI shape: (520, 17)
Columns: ['Age', 'Gender', 'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching', 'Irritability', 'delayed healing', 'partial paresis', 'muscle stiffness', 'Alopecia', 'Obesity', 'class']
Missing values:
Age                   0
Gender                0
Polyuria              0
Polydipsia            0
sudden weight loss    0
weakness              0
Polyphagia            0
Genital thrush        0
visual blurring       0
Itching               0
Irritability          0
delayed healing       0
partial paresis       0
muscle stiffness      0
Alopecia              0
Obesity               0
class                 0
dtype: int64


,Age,Gender,Polyuria,Polydipsia,sudden weight loss,weakness,Polyphagia,Genital thrush,visual blurring,Itching,Irritability,delayed healing,partial paresis,muscle stiffness,Alopecia,Obesity,class
0,40,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes,Positive
1,58,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No,Positive
2,41,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No,Positive
3,45,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No,Positive
4,60,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Positive


In [6]:
diabetes_uci['class'] = diabetes_uci['class'].astype(str).str.strip().str.lower()

diabetes_uci['diabetes_label'] = diabetes_uci['class'].apply(
    lambda x: 1 if 'positive' in x else 0
)

diabetes_uci = diabetes_uci.drop(columns=['class'])
diabetes_uci = diabetes_uci.drop_duplicates()

print("Final UCI shape:", diabetes_uci.shape)

Final UCI shape: (251, 17)


## NHANES Diabetes Files

In [21]:
ghb   = pd.read_sas(PATH_GHB)    # HbA1c
glu   = pd.read_sas(PATH_GLU)    # Fasting glucose
hdl   = pd.read_sas(PATH_HDL)    # HDL cholesterol
ins   = pd.read_sas(PATH_INS)    # Insulin
tchol = pd.read_sas(PATH_TCHOL)  # Total cholesterol
trigl = pd.read_sas(PATH_TRIGLY) # Triglycerides + LDL
demo = pd.read_sas(PATH_DEMO)

print('File shapes:')
print("GHB:", ghb.shape)
print("GLU:", glu.shape)
print("HDL:", hdl.shape)
print("INS:", ins.shape)
print("TCHOL:", tchol.shape)
print("TRIGLY:", trigl.shape)

File shapes:
GHB: (10409, 2)
GLU: (5090, 4)
HDL: (12198, 3)
INS: (5090, 5)
TCHOL: (12198, 3)
TRIGLY: (5090, 10)


In [22]:
# merge all biomarker files using participant ID
nhanes_diabetes = (
    ghb.merge(glu, on='SEQN', how='outer')
       .merge(hdl, on='SEQN', how='outer')
       .merge(ins, on='SEQN', how='outer')
       .merge(tchol, on='SEQN', how='outer')
       .merge(trigl, on='SEQN', how='outer')
)

print("\nMerged shape:", nhanes_diabetes.shape)

# Keep only variables needed for analysis
cols = [
    'SEQN',      # participant ID
    'LBXGH',     # HbA1c
    'LBDGLUSI',  # glucose
    'LBDHDD',    # HDL
    'LBXIN',     # insulin
    'LBXTC',     # total cholesterol
    'LBDLDL',    # LDL
    'LBXTR'      # triglycerides
]

cols = [c for c in cols if c in nhanes_diabetes.columns]
nhanes_diabetes = nhanes_diabetes[cols]

biomarkers = [c for c in cols if c != 'SEQN']
nhanes_diabetes = nhanes_diabetes.dropna(subset=biomarkers, how='all')

for col in biomarkers:
    if nhanes_diabetes[col].isna().sum() > 0:
        median = nhanes_diabetes[col].median()
        nhanes_diabetes[col] = nhanes_diabetes[col].fillna(median)

# Create diabetes label from HbA1c
if 'LBXGH' in nhanes_diabetes.columns:
    nhanes_diabetes['diabetes_label'] = (nhanes_diabetes['LBXGH'] >= 6.5).astype(int)

print("\nFinal shape:", nhanes_diabetes.shape)

print("\nMissing values:")
print(nhanes_diabetes.isnull().sum())

print("\nSummary statistics:")
print(nhanes_diabetes.describe())

nhanes_diabetes.head()


Merged shape: (12198, 22)

Final shape: (11053, 9)

Missing values:
SEQN              0
LBXGH             0
LBDGLUSI          0
LBDHDD            0
LBXIN             0
LBXTC             0
LBDLDL            0
LBXTR             0
diabetes_label    0
dtype: int64

Summary statistics:
                SEQN         LBXGH      LBDGLUSI        LBDHDD         LBXIN  \
count   11053.000000  11053.000000  11053.000000  11053.000000  11053.000000   
mean   117080.141048      5.733955      5.879641     53.423505     12.186699   
std      4498.086999      1.004416      1.344827     14.887953     14.964993   
min    109264.000000      2.800000      2.610000      5.000000      0.710000   
25%    113183.000000      5.300000      5.660000     43.000000     10.180000   
50%    117082.000000      5.500000      5.660000     51.000000     10.180000   
75%    120998.000000      5.800000      5.660000     61.000000     10.180000   
max    124822.000000     16.200000     29.100000    189.000000    512.500000 

,SEQN,LBXGH,LBDGLUSI,LBDHDD,LBXIN,LBXTC,LBDLDL,LBXTR,diabetes_label
0,109264.0,5.3,5.38,72.0,6.05,166.0,86.0,40.0,0
1,109266.0,5.2,5.66,56.0,10.18,195.0,101.0,84.0,0
2,109270.0,5.5,5.66,47.0,10.18,103.0,101.0,84.0,0
3,109271.0,5.6,5.72,33.0,16.96,147.0,97.0,84.0,0
4,109273.0,5.1,5.66,42.0,10.18,164.0,101.0,84.0,0


In [23]:
output_dir = "../final_cleaned_data_files"
os.makedirs(output_dir, exist_ok=True)

diabetes.to_csv(os.path.join(output_dir, "diabetes_cleaned.csv"), index=False)
nhanes_diabetes.to_csv(os.path.join(output_dir, "nhanes_diabetes_cleaned.csv"), index=False)

print("Saved:")
print(" diabetes_cleaned.csv")
print(" nhanes_diabetes_cleaned.csv")

Saved:
 diabetes_cleaned.csv
 nhanes_diabetes_cleaned.csv
